# HLS4ML Model Synthesis Notebook

In [ ]:
import hls4ml
import torch

from embedding.utils.cfg_handler import train_config, data_config
from embedding.utils.data_utils import load_data, delta_r_from_normalized
from embedding.dataloader import PFCandsDataset
from embedding.preprocs import PFPreProcessor
from embedding.models import TransformerEncoder, Projector

from pprint import pprint

train_cfg = train_config('/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/configs/train_config_hlt.yaml')
data_cfg = data_config('/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/configs/data_config_smcocktail.yaml')
test_data_path = '/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/data/embedding_hlt_smcocktail_test.pt'
model_path = '/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/models/embedding_hlt_linformer_encoder_20260406_211840.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
feature_block, label_block, flags = load_data(test_data_path, map_location=device, max_events=100)
dataset = PFCandsDataset(feature_block, label_block, device=device)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)

In [ ]:
model_artifacts = torch.load(model_path, map_location=device)
hps = model_artifacts['train_cfg']['hyperparameters']

preproc = PFPreProcessor(norm_constants={}).to(device).eval() # {} -> no norm constants
encoder = TransformerEncoder(
    preproc.num_features,
    hps["embed_size"], 
    hps["latent_dim"], 
    num_heads=hps["num_heads"],
    num_layers=hps["num_layers"],
    linear_dim=hps["linear_dim"], 
    num_tokens=feature_block.size(1),
    pairwise=False
).to(device).eval()
projector = Projector(hps["latent_dim"], hps["proj_dim"], hidden_dim=(hps["proj_dim"]*4)).to(device).eval()

preproc.load_state_dict(model_artifacts["preproc"])
encoder.load_state_dict(model_artifacts["encoder"])
projector.load_state_dict(model_artifacts["projector"])
norm_constants = model_artifacts["norm_constants"]

In [ ]:
class Model(torch.nn.Module):
    def __init__(self, encoder, projector, device):
        super().__init__()
        self.encoder = encoder
        self.projector = projector
        self.device = device
    
    def forward(self, x, mask):
        # x should be preprocessed
        mask = torch.cat(
            [
                torch.zeros(mask.size(0), 1, device=mask.device, dtype=torch.bool),
                mask.bool()
            ], dim=1
        )
        x = self.encoder(x, None, mask)
        x = self.projector(x)
        return x

In [ ]:
# Model inference
model = Model(encoder, projector, device).to(device).eval()
with torch.no_grad():
    for x, mask, y in dataloader:
        x = x.to(device)
        x = preproc(x)
        mask = mask.to(device)
        y = y.to(device)
        out = model(x, mask)
        break
print(x.shape)
print(mask.shape)
print(out.shape)

In [ ]:
torch.onnx.export(
    model,
    (
        torch.randn(x.shape, device=device),
        torch.randint(0, 2, mask.shape, device=device).bool()
    ),
    'embedding_model.onnx',
    export_params=True,
    input_names=['x', 'mask'],
    output_names=['embeddings'],
    dynamic_axes={
        'x': {0: 'batch_size'},
        'mask': {0: 'batch_size'},
        'embeddings': {0: 'batch_size'}
    }
)

config = hls4ml.utils.config_from_onnx_model('embedding_model.onnx', granularity='name')
# config['Model']['Precision'] = 'ap_fixed<16,6>'
# config['Model']['ReuseFactor'] = 1
# config['Model']['Strategy'] = 'Latency'
pprint(config)
print()

hls_model = hls4ml.converters.convert_from_onnx_model(
    model='embedding_model.onnx',
    output_dir='hls4ml_prj',
    project_name='project',
    backend='Vivado',
    hls_config=config,
    part='xcvu13p-fsga2577-2-e',
    clock_period=5,
    io_type='io_parallel'
)